In [2]:
import numpy as np
import pandas as pd

from math import floor
from tqdm import tqdm

In [3]:
gear = pd.read_csv("gear.csv").replace(np.nan, 0)

In [4]:
gear["max_stat"] = [gear.iloc[i][["crit", "direct", "det"]].max() for i in range(len(gear))]
gear

,name,slot,crit,direct,det,speed,crit materia,direct materia,det materia,max_stat
0,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,0.0,0.0,427.0
1,Grand Champion's Coat of Scouting,Body,0.0,299.0,427.0,0.0,0.0,0.0,0.0,427.0
2,Augmented Bygone Brass Bracelet of Aiming,Bracelets,212.0,148.0,0.0,0.0,0.0,0.0,0.0,212.0
3,Grand Champion's Bracelets of Aiming,Bracelets,0.0,212.0,148.0,0.0,0.0,0.0,0.0,212.0
4,Grand Champion's Ear Cuff of Aiming,Earrings,212.0,0.0,148.0,0.0,0.0,0.0,0.0,212.0
5,Augmented Bygone Brass Earrings of Aiming,Earrings,148.0,212.0,0.0,0.0,0.0,0.0,0.0,212.0
6,Augmented Bygone Brass Sabatons of Scouting,Feet,188.0,269.0,0.0,0.0,0.0,0.0,0.0,269.0
7,Grand Champion's Sabatons of Scouting,Feet,0.0,269.0,0.0,188.0,0.0,0.0,0.0,269.0
8,Augmented Bygone Brass Clawtips of Scouting,Hands,188.0,0.0,269.0,0.0,0.0,0.0,0.0,269.0
9,Grand Champion's Gloves of Scouting,Hands,0.0,0.0,269.0,188.0,0.0,0.0,0.0,269.0


In [5]:
new_slots = []

for slot_index in range(len(gear)):
    slot = gear.iloc[slot_index]

    # crit + crit
    temp1 = slot.copy()
    temp1["crit materia"] = min(108, temp1["max_stat"] - temp1["crit"])
    new_slots.append(temp1)

    # crit + direct
    temp2 = slot.copy()
    temp2["crit materia"] = min(54, temp2["max_stat"] - temp2["crit"])
    temp2["direct materia"] = min(54, temp2["max_stat"] - temp2["direct"])
    new_slots.append(temp2)

    # crit + det
    temp3 = slot.copy()
    temp3["crit materia"] = min(54, temp3["max_stat"] - temp3["crit"])
    temp3["det materia"] = min(54, temp3["max_stat"] - temp3["det"])
    new_slots.append(temp3)

    # direct + direct
    temp4 = slot.copy()
    temp4["direct materia"] = min(108, temp4["max_stat"] - temp4["direct"])
    new_slots.append(temp4)

    # direct + det
    temp5 = slot.copy()
    temp5["direct materia"] = min(54, temp5["max_stat"] - temp5["direct"])
    temp5["det materia"] = min(54, temp5["max_stat"] - temp5["det"])
    new_slots.append(temp5)

    # det + det
    temp6 = slot.copy()
    temp6["det materia"] = min(108, temp6["max_stat"] - temp6["det"])
    new_slots.append(temp6)

In [15]:
gear = pd.DataFrame(new_slots).reset_index(drop=True)
gear.to_csv("gear_materia.csv")
gear

,name,slot,crit,direct,det,speed,crit materia,direct materia,det materia,max_stat
0,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,0.0,0.0,427.0
1,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,54.0,0.0,427.0
2,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,0.0,54.0,427.0
3,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,108.0,0.0,427.0
4,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,54.0,54.0,427.0
...,...,...,...,...,...,...,...,...,...,...
91,Augmented Bygone Brass Choker of Aiming,Necklace,0.0,148.0,212.0,0.0,54.0,54.0,0.0,212.0
92,Augmented Bygone Brass Choker of Aiming,Necklace,0.0,148.0,212.0,0.0,54.0,0.0,0.0,212.0
93,Augmented Bygone Brass Choker of Aiming,Necklace,0.0,148.0,212.0,0.0,0.0,64.0,0.0,212.0
94,Augmented Bygone Brass Choker of Aiming,Necklace,0.0,148.0,212.0,0.0,0.0,54.0,0.0,212.0


In [7]:
def numberToBase(n, b, length):
    if n == 0:
        return [0] * length
    digits = []
    while n:
        digits.append(int(n % b))
        n //= b
    return [0] * (length - len(digits)) + digits[::-1]

numberToBase(100, 12, 8)

[0, 0, 0, 0, 0, 0, 8, 4]

In [8]:
def dps_increase(crit, direct, det):

    direct_chance = floor((direct - 420) / 5.055,) / 1000
    crit_chance = floor((crit - 420) / 13.9) / 1000 + 0.05
    crit_damage = 1 + floor((crit - 420) / 13.9) / 1000 + 0.4
    det_damage = 1 + floor((det - 440) / 19.9) / 1000
    
    normal_hit_weight = (1 - crit_chance) * (1 - direct_chance)
    direct_hit_weight = (1 - crit_chance) * direct_chance * 1.25
    crit_weight = crit_chance * (1 - direct_chance) * crit_damage
    direct_crit_weight = crit_chance * direct_chance * 1.25 * crit_damage
    
    DPSINC = normal_hit_weight + direct_hit_weight + crit_weight + direct_crit_weight
    DPSINC = det_damage * DPSINC

    return DPSINC

In [9]:
gear_indexes = [[12*i + k for k in range(12)] for i in range(8)]
gear_indexes

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
 [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23],
 [24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35],
 [36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47],
 [48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
 [60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71],
 [72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83],
 [84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95]]

In [10]:
gear

,name,slot,crit,direct,det,speed,crit materia,direct materia,det materia,max_stat
0,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,0.0,0.0,427.0
1,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,54.0,0.0,427.0
2,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,0.0,54.0,427.0
3,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,108.0,0.0,427.0
4,Augmented Bygone Brass Jacket of Scouting,Body,427.0,299.0,0.0,0.0,0.0,54.0,54.0,427.0
...,...,...,...,...,...,...,...,...,...,...
91,Augmented Bygone Brass Choker of Aiming,Necklace,0.0,148.0,212.0,0.0,54.0,54.0,0.0,212.0
92,Augmented Bygone Brass Choker of Aiming,Necklace,0.0,148.0,212.0,0.0,54.0,0.0,0.0,212.0
93,Augmented Bygone Brass Choker of Aiming,Necklace,0.0,148.0,212.0,0.0,0.0,64.0,0.0,212.0
94,Augmented Bygone Brass Choker of Aiming,Necklace,0.0,148.0,212.0,0.0,0.0,54.0,0.0,212.0


In [19]:
max_dps_inc = 1
max_dps_comb = []

for n in tqdm(range(12 ** 8)):
    comb = numberToBase(n, 12, 8)
    gear_comb = [12 * i + comb[i] for i in range(8)]

    sums = gear.iloc[gear_comb].sum()[["crit", "direct", "det", "crit materia", "direct materia", "det materia"]].values

    crit = sums[0] + sums[3] + 832 + 212 + 148
    direct = sums[1] + sums[4] + 854 + 148
    det = sums[2] + sums[5] + 440 + 212
    dps_inc = dps_increase(
        crit,
        direct,
        det
    )

    if dps_inc > max_dps_inc:
        print("NEW BEST:", gear_comb, dps_inc)

        max_dps_inc = dps_inc
        max_dps_comb = gear_comb

  0%|          | 303/429981696 [00:00<79:08:10, 1509.29it/s]

[0, 12, 24, 36, 48, 60, 72, 84]
1.3235587202499999
[0, 12, 24, 36, 48, 60, 72, 85]
1.3269327980999999
[0, 12, 24, 36, 48, 60, 73, 85]
1.33030687595
[0, 12, 24, 36, 48, 60, 74, 85]
1.3307204274000002
[0, 12, 24, 36, 48, 60, 75, 85]
1.3333742194499998
[0, 12, 24, 36, 48, 60, 76, 85]
1.3341041363000001
[0, 12, 24, 36, 48, 60, 77, 85]
1.3345080567
[0, 12, 24, 36, 48, 61, 75, 85]
1.3367482972999998
[0, 12, 24, 36, 48, 61, 76, 85]
1.3371802353
[0, 12, 24, 36, 48, 61, 77, 85]
1.33790139665


  0%|          | 767/429981696 [00:00<77:37:01, 1538.82it/s]

[0, 12, 24, 36, 48, 63, 75, 85]
1.3401223751499998
[0, 12, 24, 36, 48, 63, 76, 85]
1.3405639442
[0, 12, 24, 36, 48, 63, 77, 85]
1.3409862511499997


  0%|          | 2478/429981696 [00:01<77:51:19, 1534.11it/s]

[0, 12, 24, 36, 49, 63, 75, 85]
1.34145255228
[0, 12, 24, 36, 49, 63, 76, 85]
1.341903065956
[0, 12, 24, 36, 49, 63, 77, 85]
1.342334346836


  0%|          | 22732/429981696 [00:15<79:39:36, 1499.28it/s]


KeyboardInterrupt: 

In [ ]:
gear.iloc[max_dps_comb]